# Deep Learning Intro (From Scratch)

Начинаем максимально просто и пошагово, без `torch`, `tensorflow`, `keras`.

План:
1. Создаем **игрушечный датасет**
2. Понимаем, что такое **dot-умножение**
3. Обучаем 1 нейрон вручную
4. Показываем **backpropagation** и **gradient descent** на формулах и в коде

In [1]:
import numpy as np

# Для воспроизводимости результата.
np.random.seed(42)

In [2]:
# 1) Игрушечный датасет
# Истинная зависимость:
# y = 2*x1 - 3*x2 + 1 + noise

def create_toy_dataset(n_samples: int = 200):
    X = np.random.randn(n_samples, 2)
    noise = 0.1 * np.random.randn(n_samples, 1)
    y = 2.0 * X[:, [0]] - 3.0 * X[:, [1]] + 1.0 + noise
    return X, y


X, y = create_toy_dataset(8)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nПервые 3 строки X:\n", X[:3])
print("\nПервые 3 значения y:\n", y[:3])

X shape: (8, 2)
y shape: (8, 1)

Первые 3 строки X:
 [[ 0.49671415 -0.1382643 ]
 [ 0.64768854  1.52302986]
 [-0.23415337 -0.23413696]]

Первые 3 значения y:
 [[ 2.3069381 ]
 [-2.24228776]
 [ 1.14330171]]


In [3]:
X

array([[ 0.49671415, -0.1382643 ],
       [ 0.64768854,  1.52302986],
       [-0.23415337, -0.23413696],
       [ 1.57921282,  0.76743473],
       [-0.46947439,  0.54256004],
       [-0.46341769, -0.46572975],
       [ 0.24196227, -1.91328024],
       [-1.72491783, -0.56228753]])

In [4]:
y

array([[ 2.3069381 ],
       [-2.24228776],
       [ 1.14330171],
       [ 1.71489107],
       [-1.42006403],
       [ 1.44777625],
       [ 7.2305181 ],
       [-0.9054479 ]])

In [8]:
# 2) Что такое dot-умножение
# Для векторов: [a1, a2] · [b1, b2] = a1*b1 + a2*b2

a = np.array([1.0, 2.0])
b = np.array([3.0, 4.0])


print("a · b =", np.dot(a, b))  # 1*3 + 2*4 = 11

# Для матриц (batch входов) и весов одного нейрона:
# X shape: (batch, features)
# w shape: (features, 1)
# X.dot(w) -> (batch, 1)
X_demo = np.array([[1.0, 2.0], [5.0, 6.0]])
w_demo = np.array([[0.3], [0.7]])

print("\nX_demo:\n", X_demo)
print("\nw_demo:\n", w_demo)
print("\nX_demo.dot(w_demo):\n", X_demo.dot(w_demo))

a · b = 11.0

X_demo:
 [[1. 2.]
 [5. 6.]]

w_demo:
 [[0.3]
 [0.7]]

X_demo.dot(w_demo):
 [[1.7]
 [5.7]]


## 3) Backpropagation + Gradient Descent на 1 нейроне

**Модель (линейный нейрон):**

$$
\hat{y} = Xw + b
$$

где:

$$
X \in \mathbb{R}^{N \times d}, \quad w \in \mathbb{R}^{d \times 1}, \quad b \in \mathbb{R}^{1 \times 1}, \quad \hat{y} \in \mathbb{R}^{N \times 1}
$$

**Функция потерь (MSE):**

$$
L = \frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i - y_i)^2
$$

Обозначим:

$$
\mathrm{diff} = \hat{y} - y
$$

**Градиенты:**

$$
\frac{\partial L}{\partial \hat{y}} = \frac{2}{N}(\hat{y} - y)
$$

$$
\frac{\partial L}{\partial w} = X^T \frac{\partial L}{\partial \hat{y}}
$$

$$
\frac{\partial L}{\partial b} = \sum_{i=1}^{N} \frac{\partial L}{\partial \hat{y}_i}
$$

Идея: считаем градиенты и обновляем параметры против градиента, чтобы уменьшать ошибку.

In [6]:
def mse_loss(y_pred, y_true):
    """
    Возвращает:
    - loss: скаляр (MSE)
    - grad_y_pred: градиент dL/dy_pred формы (N, 1)
    """
    diff = y_pred - y_true
    loss = np.mean(diff ** 2)
    grad_y_pred = (2.0 / y_true.shape[0]) * diff
    return loss, grad_y_pred

### Интуиция Backpropagation (очень просто)

Мы хотим уменьшить ошибку $L$.

1. Сначала считаем, как меняется ошибка при изменении предсказания:

$$
\frac{\partial L}{\partial \hat{y}}
$$

2. Через правило цепочки переносим этот сигнал к параметрам:

$$
\frac{\partial L}{\partial w} = X^T \frac{\partial L}{\partial \hat{y}}, \qquad
\frac{\partial L}{\partial b} = \sum \frac{\partial L}{\partial \hat{y}}
$$

3. Делаем шаг градиентного спуска:

$$
w \leftarrow w - \eta \frac{\partial L}{\partial w}, \qquad
b \leftarrow b - \eta \frac{\partial L}{\partial b}
$$

где $\eta$ (`learning rate`) — размер шага.

In [11]:
X, y = create_toy_dataset(200)

w = 0.1 * np.random.randn(2, 1)
b = np.zeros((1, 1))

print("Формы тензоров:")
print("X:", X.shape, "y:", y.shape, "w:", w.shape, "b:", b.shape)

learning_rate = 0.1
epochs = 100

for epoch in range(1, epochs + 1):
    y_pred = X.dot(w) + b
    loss, grad_y_pred = mse_loss(y_pred, y)

    grad_w = X.T.dot(grad_y_pred)
    grad_b = np.sum(grad_y_pred, axis=0, keepdims=True)

    w -= learning_rate * grad_w
    b -= learning_rate * grad_b

    print(f"Epoch {epoch:3d} | Loss: {loss:.6f}")

print("\nОбучение завершено")
print("w:\n", w)
print("b:\n", b)
print("Ожидаем w примерно [2, -3], b примерно 1")

Формы тензоров:
X: (200, 2) y: (200, 1) w: (2, 1) b: (1, 1)
Epoch   1 | Loss: 12.312444
Epoch   2 | Loss: 8.124909
Epoch   3 | Loss: 5.382427
Epoch   4 | Loss: 3.579432
Epoch   5 | Loss: 2.389614
Epoch   6 | Loss: 1.601549
Epoch   7 | Loss: 1.077720
Epoch   8 | Loss: 0.728331
Epoch   9 | Loss: 0.494524
Epoch  10 | Loss: 0.337572
Epoch  11 | Loss: 0.231898
Epoch  12 | Loss: 0.160549
Epoch  13 | Loss: 0.112248
Epoch  14 | Loss: 0.079470
Epoch  15 | Loss: 0.057174
Epoch  16 | Loss: 0.041976
Epoch  17 | Loss: 0.031595
Epoch  18 | Loss: 0.024492
Epoch  19 | Loss: 0.019624
Epoch  20 | Loss: 0.016282
Epoch  21 | Loss: 0.013984
Epoch  22 | Loss: 0.012403
Epoch  23 | Loss: 0.011313
Epoch  24 | Loss: 0.010561
Epoch  25 | Loss: 0.010041
Epoch  26 | Loss: 0.009682
Epoch  27 | Loss: 0.009434
Epoch  28 | Loss: 0.009262
Epoch  29 | Loss: 0.009143
Epoch  30 | Loss: 0.009060
Epoch  31 | Loss: 0.009003
Epoch  32 | Loss: 0.008963
Epoch  33 | Loss: 0.008936
Epoch  34 | Loss: 0.008917
Epoch  35 | Loss: 0.0

## Что дальше

Когда это станет понятно, можно добавить:
1. Мини-батчи (stochastic/mini-batch gradient descent)
2. Нелинейность (`ReLU`) и 2-слойную сеть
3. Бинарную классификацию (`sigmoid` + `BCE`)
4. Разделение на `train/test` и метрики